# Geospatial Data Analysis Lab: Steel Plants Dataset


**Learning Objectives:**
- Perform exploratory data analysis (EDA) on geospatial datasets
- Visualize geospatial data using interactive maps with Plotly
- Merge granular exposure / population data (LitPop) with asset locations
- Aggregate data at the company level
- Integrate geospatial visualizations into a Streamlit dashboard

**Dataset:** Download the Global Energy Monitor [Global Iron and Steel Tracker](https://globalenergymonitor.org/projects/global-iron-steel-tracker) (formerly Global Steel Plant Tracker). Use the plant-level download from that page and place the file in the same folder as this notebook (or update the load path accordingly).

---


## Submission info

Work in **groups of up to 4**. Fill in every member before submitting.

| # | Full name | Student ID |
|---|-----------|------------|
| 1 | Charlotte Le Bihan | B00818310 |
| 2 | Ines Lebard | B00820964 |
| 3 | Camille Tanguy | B00821659 |
| 4 | Oceane Tanios | B00822694 |

**Group / repo name:** `aidams-lab1-<surname1>-<surname2>-...`  
**Submitter (one person):**  
**Repo URL:**  
**Streamlit Cloud URL (bonus):**  

### What to submit
- This notebook (`lab_1.ipynb`) with all parts completed and cells run
- `app.py` (Part 6)
- Processed data exports used by the dashboard (e.g. CSV/Parquet), if applicable
- (Bonus) Deployed Streamlit Cloud app link, if completed

<span style="color: #FFD700; font-weight: bold">Send submission info to my email (1 email per group)</span> — include the GitHub repo URL and, if you did the bonus, the Streamlit Cloud link.


## Upload to GitHub

Follow this checklist (one repo per group):

1. Create a **private** repository (or use the course organization if provided).
2. Name it using the pattern above, e.g. `aidams-lab1-ali-ben-chen-diaz`.
3. Add your files (`lab_1.ipynb`, `app.py`, exports, and a short `README.md` with how to run the dashboard).
5. Commit and push:
   ```bash
   git init
   git add lab_1.ipynb app.py .gitignore README.md
   git commit -m "Complete AIDAMS Lab 1"
   git branch -M main
   git remote add origin <YOUR_REPO_URL>
   git push -u origin main
   ```
6. Invite the instructor (or open the assignment link) and paste the **repo URL** in the Submission info table above.

**Done when:** all 4 names are filled in, the notebook contains all outputs (no need for the instructor to re-run it), maps are visible, and `streamlit run app.py` works from the repo.

**Bonus (optional):** deploy `app.py` to Streamlit Cloud and paste the public app URL above / in your submission email.


## Part 1: Setup and Data Loading

Import the necessary libraries and load the steel plants dataset.

**Tip:** After loading, run `df.columns` and `df.head()`. Column names in the file may differ slightly by release — inspect them and adapt your code accordingly.


In [13]:
# Import required libraries
# - pandas for data manipulation
# - numpy for numerical operations
# - plotly.express for interactive visualizations
# - Any other libraries you need

import pandas as pd
import numpy as np
import plotly.express as px

In [14]:
# Load the steel plants dataset
# Tip: start with df.columns / df.head() and adapt names if your file differs slightly
#
# Common columns in this steel plant dataset:
# - Plant name (English)
# - Owner
# - Country/Area, Region
# - Coordinates  (often a single "lat, lon" string — not separate latitude/longitude columns)
# - Plant age (years)
# - Capacity field: usually "Nominal crude steel capacity (ttpa)" (check your df.columns to confirm the exact name)
#   (Older datasets may also include additional fields like ferronickel/sinter/coking/pelletizing capacities, 
#    but most analyses focus on nominal crude steel capacity as the main output.)

FILE = "Plant-level_data_Global_Iron_and_Steel_Tracker_June_2026_V1.xlsx"

plants = pd.read_excel(FILE, sheet_name="Plant data")

# Load capacity sheet and convert the capacity column to numbers (some cells contain text like ">0")
capacity = pd.read_excel(FILE, sheet_name="Plant capacities and status")
capacity["Nominal crude steel capacity (ttpa)"] = pd.to_numeric(
    capacity["Nominal crude steel capacity (ttpa)"], errors="coerce"  # non-numeric values like ">0" become NaN
)
# Keep only active production units so retired/cancelled/mothballed capacity is not included
ACTIVE_STATUSES = {"operating", "construction", "announced", "operating pre-retirement"}
df_cap = (
    capacity[capacity["Status"].isin(ACTIVE_STATUSES)]
    .groupby("GEM plant ID", as_index=False)["Nominal crude steel capacity (ttpa)"]
    .sum(min_count=1)  # min_count=1 preserves NaN for plants whose only entries were all-NaN
)

df = plants.merge(df_cap, on="GEM plant ID", how="left")

print(df.shape)
df.head()


(1293, 45)


,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Steel sector end users,Workforce size,ISO 14001,ISO 50001,ResponsibleSteel certification,Main production equipment,Power source,Iron ore source,Met coal source,Nominal crude steel capacity (ttpa)
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,unknown,900,unknown,unknown,no,EAF,unknown,unknown,NaN,1100.0
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,building and infrastructure,5500,unknown,unknown,no,EAF,unknown,unknown,NaN,3000.0
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,unknown,4500,2025-10-06 00:00:00,unknown,no,EAF,unknown,NaN,unknown,1600.0
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"building and infrastructure, energy",unknown,unknown,unknown,no,EAF,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown,1400.0
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,"automotive, building and infrastructure, energ...",11000,2025-04-30 00:00:00,2025-12-04 00:00:00,no,DRI; EAF; BF; BOF,unknown,unknown,unknown,13800.0


---
## Part 2: Exploratory Data Analysis

Answer the following questions through your analysis:


### Question 1: Data Overview
**Task:** Display basic information about the dataset.
- How many steel plants are in the dataset?
- What are the column names and data types? (use `df.columns` / `df.dtypes` or `df.info()`, then adapt later code to the names you see)
- Are there any missing values?


In [ ]:
# Display dataset shape
print(f"Number of steel plants: {len(df)}")
print(f"Number of columns: {df.shape[1]}")
df.shape

In [16]:
# Display column information and data types
# Start here: print(df.columns) and adapt column names in later cells if needed
print(df.columns.tolist())
print()
df.info()

['GEM plant ID', 'Plant name (English)', 'Plant name (other language)', 'Other plant names (English)', 'Other plant names (other language)', 'Owner', 'Owner (other language)', 'Owner GEM entity ID', 'Owner PermID', 'SOE status', 'Parent (English)', 'Parent GEM entity ID', 'Parent PermID', 'Location address', 'Location address (other language)', 'Municipality', 'Subnational unit', 'Country/area', 'Region', 'Coordinates', 'Coordinate accuracy', 'GEM wiki page', 'Plant age', 'Announced date', 'Construction date', 'Start date', 'Pre-retirement announcement date', 'Idled date', 'Retired date', 'Ferronickel capacity (ttpa)', 'Sinter plant capacity (ttpa)', 'Coking plant capacity (ttpa)', 'Pelletizing plant capacity (ttpa)', 'Category steel product', 'Steel products', 'Steel sector end users', 'Workforce size', 'ISO 14001', 'ISO 50001', 'ResponsibleSteel certification', 'Main production equipment', 'Power source', 'Iron ore source', 'Met coal source', 'Nominal crude steel capacity (ttpa)']

<

In [17]:
# Check for missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(missing)} / {df.shape[1]}")
print(f"Total missing cells: {int(df.isnull().sum().sum())}")
missing

Columns with missing values: 14 / 45
Total missing cells: 7435


SOE status                             1081
Other plant names (other language)      958
Location address (other language)       794
Owner (other language)                  715
Ferronickel capacity (ttpa)             694
Coking plant capacity (ttpa)            689
Other plant names (English)             551
Pelletizing plant capacity (ttpa)       528
Sinter plant capacity (ttpa)            512
Plant name (other language)             502
Nominal crude steel capacity (ttpa)     219
Met coal source                         115
Plant age                                61
Iron ore source                          16
dtype: int64

### Question 2: Statistical Summary
**Task:** Generate descriptive statistics for numerical columns.
- What is the average plant capacity? (sum relevant capacity columns in ttpa if needed, e.g. sinter / coking / pelletizing / ferronickel)
- What is the range of latitudes and longitudes? (you will likely need to parse `Coordinates` first — see Part 3 hint)
- What is the distribution of `Plant age (years)`?


In [ ]:
# Parse Coordinates into Latitude and Longitude
def parse_coordinates(coord_str):
    if pd.isna(coord_str):
        return (np.nan, np.nan)
    try:
        lat, lon = coord_str.split(',')
        return (float(lat.strip()), float(lon.strip()))
    except:
        return (np.nan, np.nan)

df[['Latitude', 'Longitude']] = df['Coordinates'].apply(
    lambda x: pd.Series(parse_coordinates(x))
)

# Capacity statistics
print("Average capacity:", df['Nominal crude steel capacity (ttpa)'].mean())
print("Min capacity:", df['Nominal crude steel capacity (ttpa)'].min())
print("Max capacity:", df['Nominal crude steel capacity (ttpa)'].max())
print()

# Geographic range
print("Latitude range:", df['Latitude'].min(), "to", df['Latitude'].max())
print("Longitude range:", df['Longitude'].min(), "to", df['Longitude'].max())
print()

# Plant age distribution
# Convert to numeric since it might have mixed types
df['Plant age'] = pd.to_numeric(df['Plant age'], errors='coerce')
print("Average plant age:", df['Plant age'].mean())
print("Min age:", df['Plant age'].min())
print("Max age:", df['Plant age'].max())
print()

# Summary stats
df[['Nominal crude steel capacity (ttpa)', 'Latitude', 'Longitude', 'Plant age']].describe()

# Visualize plant age distribution
fig = px.histogram(df, x='Plant age', nbins=30, title='Distribution of Plant Age')
fig.show()

### Question 3: Geographic Distribution
**Task:** Analyze the geographic distribution of steel plants.
- Which `Country/Area` or `Region` values have the most steel plants?
- What is the distribution of plants by `Owner` (company)?


In [ ]:
# Count plants by country/region
def count_plants_by_region(df):
    return df.groupby("Country/area")["Plant name (English)"].count().sort_values(ascending=False)
print(count_plants_by_region(df))

In [20]:
# Count plants by Owner (company)
def count_plants_by_owner(df):
    return df.groupby("Owner")["Plant name (English)"].count().sort_values(ascending=False)
print(count_plants_by_owner(df))

Owner
Nucor Corp                                                       13
Cleveland-Cliffs Inc                                             12
Nippon Steel Corp                                                10
Commercial Metals Co                                              8
Steel Authority of India Ltd                                      8
                                                                 ..
Hebei Huaxin Special Steel Co Ltd                                 1
Hebei Jingdong Pipe Industry Co Ltd                               1
Hebei Jinxi Iron & Steel Group Co Ltd                             1
Hebei New Wuan Iron and Steel Group Xin Hui Metallurgy Co Ltd     1
Çolakoğlu Metalürji AŞ                                            1
Name: Plant name (English), Length: 1069, dtype: int64


### Question 4: Capacity Analysis
**Task:** Analyze the capacity distribution.
- What is the total global steel production capacity? (sum the capacity columns you are using)
- Which `Owner` values have the highest total capacity?
- How does capacity vary by `Region` or `Country/Area`?


In [21]:
# Calculate total capacity
def total_capacity(df):
    return df["Nominal crude steel capacity (ttpa)"].sum()
print(f"Total nominal crude steel capacity (ttpa): {total_capacity(df)}")

Total nominal crude steel capacity (ttpa): 3059924.0


In [ ]:
# Group by Owner and sum capacity
def capacity_by_owner(df):
    return df.groupby("Owner")["Nominal crude steel capacity (ttpa)"].sum().sort_values(ascending=False)
print(capacity_by_owner(df))

# Capacity by Country/area
cap_by_country = (
    df.groupby("Country/area")["Nominal crude steel capacity (ttpa)"]
    .sum()
    .sort_values(ascending=False)
)
print("\nTop 10 countries by total capacity (ttpa):")
print(cap_by_country.head(10))

# Capacity by Region
cap_by_region = (
    df.groupby("Region")["Nominal crude steel capacity (ttpa)"]
    .sum()
    .sort_values(ascending=False)
)
print("\nCapacity by region (ttpa):")
print(cap_by_region)

---
## Part 3: Geospatial Visualization with Plotly

Create interactive maps to visualize the steel plants' locations and characteristics.


### Exercise 1: Basic Scatter Map
**Task:** Create a scatter map showing all steel plant locations.
- Parse `Coordinates` into numeric `Latitude` and `Longitude` (see hint in the code cell)
- Color points by `Country/Area` or `Region`
- Add hover information showing `Plant name (English)`, `Owner`, and capacity


In [23]:
# Exercise 1: basic scatter map showing all steel plant locations

# re-parse coordinates into lowercase columns (exercises 2 and 3 also use these)
coords = df['Coordinates'].str.split(',', n=1, expand=True)
df['latitude']  = pd.to_numeric(coords[0].str.strip(), errors='coerce')
df['longitude'] = pd.to_numeric(coords[1].str.strip(), errors='coerce')

# drop plants with missing coordinates before plotting
plot_df = df.dropna(subset=['latitude', 'longitude'])

# create a world map with one dot per plant, colored by country
fig = px.scatter_geo(
    plot_df,
    lat='latitude',
    lon='longitude',
    color='Country/area',
    hover_name='Plant name (English)',
    hover_data={
        'Owner': True,
        'Nominal crude steel capacity (ttpa)': ':,.0f',
        'latitude': False,   # already visible via the map, no need to repeat
        'longitude': False,
    },
    projection='natural earth',
    title='Steel plant locations by country',
)
fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

### Exercise 2: Sized Markers by Capacity
**Task:** Create a map where marker size represents plant capacity.
- Larger markers for higher capacity plants
- Color by `Owner`
- Include interactive hover details (`Plant name (English)`, `Country/Area`, capacity, etc.)


In [24]:
# Exercise 2: scatter map where marker size reflects plant capacity

# reuse plot_df with parsed coordinates from Exercise 1
plot_df = df.dropna(subset=['latitude', 'longitude']).copy()

# clip capacity at 1 so plants with 0 capacity still appear as a small dot
plot_df['cap_size'] = plot_df['Nominal crude steel capacity (ttpa)'].fillna(0).clip(lower=1)

fig = px.scatter_geo(
    plot_df,
    lat='latitude',
    lon='longitude',
    size='cap_size',
    size_max=40,
    color='Owner',
    hover_name='Plant name (English)',
    hover_data={
        'Country/area': True,
        'Nominal crude steel capacity (ttpa)': ':,.0f',
        'cap_size': False,   # internal sizing column, no need to show
        'latitude': False,
        'longitude': False,
    },
    projection='natural earth',
    title='Plants sized by capacity',
)
fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

### Exercise 3: Density Heatmap
**Task:** Create a density map showing concentration of steel plants.
- Use Plotly's density_mapbox to show clustering
- Identify regions with high plant density


In [25]:
# Exercise 3: density heatmap showing where steel plants are most concentrated

# build plot_df here so this cell works independently of Exercise 2
plot_df = df.dropna(subset=['latitude', 'longitude']).copy()
plot_df['cap_size'] = plot_df['Nominal crude steel capacity (ttpa)'].fillna(0).clip(lower=1)

# density_map weights each point by cap_size, so high-capacity clusters appear as hot spots
fig = px.density_map(
    plot_df,
    lat='latitude',
    lon='longitude',
    z='cap_size',        # weight by capacity so bigger plants show as hotter spots
    radius=15,           # controls how spread out each blob is
    zoom=1,
    center=dict(lat=plot_df['latitude'].mean(), lon=plot_df['longitude'].mean()),
    map_style='open-street-map',  # free tiles, no API token needed
    title='Steel plant density (weighted by capacity)',
)
fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

# hotspots: East Asia (especially China) dominates, followed by Europe and South/East Asia


---
## Part 4: Merging Exposure / Population Data with Assets

Steel plants sit in real places -- next to people, housing, and economic activity. In this part you will attach **granular socio-economic exposure data** to each plant so you can ask: *who and what is near this industrial asset?*

We use **LitPop** (ETH Zurich): a global dataset that combines **population** and **produced capital / asset value** on a fine geographic grid. It is widely used in disaster- and climate-risk analysis as a measure of **exposure**. It is **not** classical environmental monitoring (not air quality, emissions, or weather).

**Goal:** spatially link LitPop grid cells (or sample points) to steel plant locations (nearest neighbor or spatial join), then use the merged fields in maps and later company-level summaries.


### Exercise 1: Load LitPop (Exposure) Data
**Task:** Load the LitPop sample and inspect it.

- **Samples for this lab (recommended):** LitPop sample files are available on Moodle (litpop data). Use these for the merge exercises below.
- **Full LitPop dataset (optional):** [ETH Research Collection – LitPop](https://www.research-collection.ethz.ch/entities/researchdata/12dcfc4f-9d03-463a-8d6b-76c0dc73cdc8)

- Expected columns (may vary by extract): location identifiers, latitude, longitude, population and/or asset-value / exposure fields, etc.

Briefly note what each column represents and the spatial resolution of the sample.


In [26]:
# Load LitPop sample (exposure / population–asset data)
# Display dataset shape
# Load LitPop sample (exposure / population-asset data)
litpop_chn = pd.read_hdf("litpop/LitPop_pc_300_arcsec_CHN_v1.hdf5")
litpop_ind = pd.read_hdf("litpop/LitPop_pc_300_arcsec_IND_v1.hdf5")
litpop_jpn = pd.read_hdf("litpop/LitPop_pc_300_arcsec_JPN_v1.hdf5")

# Combiner les trois pays en un seul DataFrame
litpop = pd.concat([litpop_chn, litpop_ind, litpop_jpn], ignore_index=True)

litpop.head()

,value,latitude,longitude,geometry,region_id,impf_
0,5.280440e+09,20.041667,110.208333,POINT (110.20833333 20.04166667),156,1
1,4.040559e+07,20.041667,110.625000,POINT (110.625 20.04166667),156,1
2,4.190224e+07,20.041667,110.708333,POINT (110.70833333 20.04166667),156,1
3,8.813872e+07,19.958333,109.541667,POINT (109.54166667 19.95833333),156,1
4,1.879947e+08,19.958333,109.625000,POINT (109.625 19.95833333),156,1


In [27]:
# Inspect LitPop data (columns, dtypes, missing values, value ranges)
# Inspect LitPop data (columns, dtypes, missing values, value ranges)
print("Shape:", litpop.shape)
print("\nColonnes:", litpop.columns.tolist())
print("\nTypes de données:")
print(litpop.dtypes)
print("\nValeurs manquantes:")
print(litpop.isnull().sum())
print("\nStatistiques descriptives:")
litpop.describe()

Shape: (182591, 6)

Colonnes: ['value', 'latitude', 'longitude', 'geometry', 'region_id', 'impf_']

Types de données:
value        float64
latitude     float64
longitude    float64
geometry      object
region_id      int64
impf_          int64
dtype: object

Valeurs manquantes:
value        0
latitude     0
longitude    0
geometry     0
region_id    0
impf_        0
dtype: int64

Statistiques descriptives:


,value,latitude,longitude,region_id,impf_
count,1.825910e+05,182591.000000,182591.000000,182591.000000,182591.0
mean,3.817856e+08,33.585715,99.540705,207.031891,1.0
std,5.670982e+09,8.816306,17.568848,88.645570,0.0
min,0.000000e+00,6.875000,68.208333,156.000000,1.0
25%,5.023580e+04,27.041667,83.875000,156.000000,1.0
50%,1.295843e+06,33.875000,98.708333,156.000000,1.0
75%,1.401763e+07,40.375000,113.875000,156.000000,1.0
max,5.044057e+11,53.541667,145.791667,392.000000,1.0


### Exercise 2: Spatial Join or Nearest Neighbor Matching
**Task:** Attach LitPop exposure attributes to each steel plant based on geographic proximity.
- Match each plant to the **nearest LitPop grid cell / sample point** (or use a spatial join if you work with polygons)
- Consider `geopandas`, a ball-tree / KD-tree nearest-neighbor search, or haversine distances
- Keep plant identifiers and the LitPop fields you will use later (e.g. population, asset value / exposure)

You should end up with one row per plant (or a clear many-to-one rule if you aggregate nearby cells).


In [28]:
# Calculate distances or perform spatial join
# Hint: You might calculate haversine distance or use a spatial library
def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of earth in kilometers. Use 3956 for miles
    return c * r

In [29]:
def merge_litpop_with_plants(litpop, plants):
    merged_data = []
    for index, plant_row in plants.iterrows():
        lat_plant = plant_row['Latitude']
        lon_plant = plant_row['Longitude']

        if pd.isna(lat_plant) or pd.isna(lon_plant):
            merged_row = plant_row.to_dict()
            merged_row.update({'litpop_value': np.nan, 'distance_to_litpop_km': np.nan})
            merged_data.append(merged_row)
            continue

        distances = haversine(lat_plant, lon_plant, litpop['latitude'], litpop['longitude'])
        nearest_idx = distances.idxmin()
        nearest_litpop = litpop.loc[nearest_idx]

        merged_row = plant_row.to_dict()
        merged_row.update({
            'litpop_value': nearest_litpop['value'],
            'distance_to_litpop_km': distances.min()
        })
        merged_data.append(merged_row)

    return pd.DataFrame(merged_data)

merged_df = merge_litpop_with_plants(litpop, df)
merged_df.head()

,GEM plant ID,Plant name (English),Plant name (other language),Other plant names (English),Other plant names (other language),Owner,Owner (other language),Owner GEM entity ID,Owner PermID,SOE status,...,Power source,Iron ore source,Met coal source,Nominal crude steel capacity (ttpa),Latitude,Longitude,latitude,longitude,litpop_value,distance_to_litpop_km
0,P100000120882,Aba Iron and Steel Payas plant,ABA DEMİR ÇELİK,"EEY Iron and Steel, Nursan Steel Payas Plant (...",NaN,ABA Çelik Demir LŞ,ABA Demir ve Çelik İthalat İhracat Ticaret,E100000131190,unknown,NaN,...,unknown,unknown,NaN,1100.0,36.747413,36.217330,36.747413,36.217330,2.938832e+02,3265.769650
1,P100000120753,Abba Steel Ohangwena steel plant,NaN,Groot Suisse Oshana plant,NaN,Abba Steel Ltd,NaN,E100001012072,unknown,NaN,...,unknown,unknown,NaN,3000.0,-17.397866,15.891022,-17.397866,15.891022,1.041774e+07,7266.295666
2,P100000120802,Abinsk Electric Steel Works,АЭМЗ,"AESW, ASW, AEMZ, AEMK",Абинский ЭлектроМеталлургический завод,Abinski Elektrometallurgicheski Zavod LLC,"ООО ""АБИНСКИЙ ЭЛЕКТРОМЕТАЛЛУРГИЧЕСКИЙ ЗАВОД""",E100000130999,5039667129,NaN,...,unknown,NaN,unknown,1600.0,44.881938,38.127510,44.881938,38.127510,6.983897e+04,2966.743029
3,P100000120020,Abul Khair Steel Sitakunda plant,আবুল খায়ের স্টিল মেল্টিং লিমিটেড,"AKS Long Steel, AKS Sitakund, AKS Melting, Abu...",NaN,Abul Khair Steel Ltd,NaN,E100000131068,5074007077,NaN,...,"100MW Power Plant, in progress 50MW solar plant",unknown,unknown,1400.0,22.472008,91.734827,22.472008,91.734827,1.449703e+07,63.401381
4,P100000120620,Acciaierie d'Italia Taranto steel plant,NaN,"ILVA Taranto steel plant (predecessor), ILVA S...",NaN,Acciaierie d'italia SpA,NaN,E100001010116,5067495106,Full,...,unknown,unknown,unknown,13800.0,40.508993,17.207589,40.508993,17.207589,2.938832e+02,4729.998653


### Exercise 3: Visualize Plants with Exposure Context
**Task:** Create a map of steel plants enriched with LitPop fields.
- Color plants by a LitPop metric (e.g. local population or asset exposure)
- Size markers by plant capacity
- Add hover details with both plant attributes (`Plant name (English)`, `Owner`, capacity) and the matched LitPop values

Interpret briefly: where do large plants sit relative to high population / high asset-value areas?


In [30]:
def plot_plants_with_litpop(merged_df):
    plot_df = merged_df.dropna(subset=['Latitude', 'Longitude']).copy()
    plot_df['cap_size'] = plot_df['Nominal crude steel capacity (ttpa)'].fillna(0).clip(lower=1)
    fig = px.scatter_geo(plot_df,
                          lat='Latitude',
                          lon='Longitude',
                          size='cap_size',
                          size_max=40,
                          color='litpop_value',
                          hover_name='Plant name (English)',
                          hover_data={
                              'Owner': True,
                              'Nominal crude steel capacity (ttpa)': ':,.0f',
                              'litpop_value': True,
                              'cap_size': False,
                          },
                          color_continuous_scale='viridis',
                          title="Steel Plants Colored by LitPop Exposure")
    fig.show()

---
## Part 5: Company-Level Aggregation

Aggregate data at the company level to analyze corporate footprints — including capacity and the LitPop exposure context you attached in Part 4.


### Exercise 1: Aggregate Metrics by Company
**Task:** Group plants by company (`Owner`) and calculate aggregate metrics.
- Total capacity per company
- Number of plants per company
- Average LitPop exposure metrics per company (from Part 4)
- Geographic spread (e.g. number of `Country/Area` or `Region` values)


In [31]:
# Group by company and aggregate
def aggregate_by_company(merged_df):
    return merged_df.groupby('Owner').agg({
        'Nominal crude steel capacity (ttpa)': 'sum',
        'litpop_value': 'mean',
        'distance_to_litpop_km': 'mean'
    }).reset_index()


### Exercise 2: Company Headquarters or Centroid
**Task:** Calculate a representative location for each company.
- Option 1: Use the centroid of all plant locations
- Option 2: Use the location of the largest plant
- Option 3: Assign actual headquarters coordinates


In [32]:
# Calculate company representative locations
def calculate_company_locations(merged_df):
    company_locations = merged_df.groupby('Owner').agg({
        'Latitude': 'mean',
        'Longitude': 'mean'
    }).reset_index()
    return company_locations


### Exercise 3: Visualize Company-Level Data
**Task:** Create a map showing companies with aggregated metrics.
- Show one marker per company at the representative location
- Size by total capacity
- Color by average LitPop exposure (or another Part 4 metric)
- Hover information with company summary statistics


In [33]:
# Create company-level visualization
def plot_companies_with_litpop(company_df):
    plot_df = company_df.dropna(subset=['Latitude', 'Longitude']).copy()
    plot_df['cap_size'] = plot_df['Nominal crude steel capacity (ttpa)'].fillna(0).clip(lower=1)
    fig = px.scatter_geo(plot_df,
                          lat='Latitude',
                          lon='Longitude',
                          size='cap_size',
                          size_max=40,
                          color='litpop_value',
                          hover_name='Owner',
                          hover_data={
                              'Nominal crude steel capacity (ttpa)': ':,.0f',
                              'litpop_value': True,
                              'cap_size': False,
                          },
                          color_continuous_scale='viridis',
                          title="Companies Colored by Average LitPop Exposure")
    fig.show()


---
## Part 6: Streamlit Dashboard Integration

Prepare your visualizations for deployment in a Streamlit dashboard.


### Exercise 1: Create Dashboard Script Structure
**Task:** Create a Streamlit app file (`app.py`) with the following structure:

```python
# Import streamlit and other necessary libraries

# Set page configuration

# Title and description

# Sidebar for filters
# - Company selector
# - Region/country filter
# - Capacity range slider

# Main content area
# - KPI metrics (total plants, total capacity, etc.)
# - Interactive map
# - Data table

# Footer with data sources and notes
```


### Exercise 1: Prepare Data for Dashboard
**Task:** Save your processed data to files that the dashboard will load.
- Export cleaned plant data
- Export merged plant + LitPop exposure data
- Export company-level aggregations
- Save as CSV or Parquet for efficient loading


In [34]:
# Save processed datasets
def save_processed_data(merged_df, company_df):
    merged_df.to_csv("merged_steel_litpop.csv", index=False)
    company_df.to_csv("company_steel_litpop.csv", index=False)


### Exercise 2: Display relevant information from your exploratory analysis into the dashboard

In [35]:
# This cell is for notes/observations about your dashboard
# What works well?
# What could be improved?
# Any performance issues with large datasets?
def optimize_data_processing(merged_df):
    # Implement optimizations such as:
    # - Reducing DataFrame size (e.g., using categorical types)
    # - Using efficient aggregations
    # - Parallel processing if applicable
    pass

---
## Bonus (optional): Deploy to Streamlit Cloud

Deploy your dashboard so it runs in the browser without local setup.

**Task:**
1. Make sure `app.py` and any required data files are in your GitHub repo
2. Go to [https://share.streamlit.io](https://share.streamlit.io) (Streamlit Community Cloud)
3. Sign in with GitHub, select your repo, and deploy `app.py`
4. Copy the public app URL

**Submit:** paste the Streamlit Cloud link in the **Submission info** section at the top **and** include it in your group's submission email (with the GitHub repo URL).


---
## Lab Summary and Key Takeaways

**What you learned:**
- How to perform EDA on geospatial datasets
- Creating interactive maps with Plotly for geospatial data
- Merging LitPop exposure / population data with assets based on geographic proximity
- Aggregating geospatial data at different levels (asset vs. company)
- Building interactive dashboards with Streamlit

**Next Steps:**
- Explore other geospatial libraries (GeoPandas, Folium, Kepler.gl)
- Learn about coordinate reference systems (CRS) and projections
- Practice with other datasets (buildings, utilities, transportation)
- (Bonus) Deploy your dashboard to Streamlit Cloud and share the link in your submission email
